In [1]:
import numpy as np
import pandas as pd
import h5py
import os
import time
import seaborn as sns
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import write_to_hdf5_file_single_array, read_in_routine, write_to_hdf5_file
from sleep_analysis_functions import compute_sleep_indices
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import itertools

from icu_sleep_breathing_2020_help_functions import * 

import matplotlib


font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 9}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 9}

matplotlib.rc('font', **font)
matplotlib.rc('font', **font)

In [2]:
def stages_to_names(stages_num):
    
    stages_names = stages_num.copy()
    stages_names[stages_num == 1] = 'N3'
    stages_names[stages_num == 2] = 'N2'
    stages_names[stages_num == 3] = 'N1'
    stages_names[stages_num == 4] = 'R'
    stages_names[stages_num == 5] = 'W'
    
    return stages_names

In [3]:
def print_hlh_metadata(lhl_path):
    with h5py.File(lhl_path, 'r') as f:
        print(list(f.keys()))

def load_hlh_data(lhl_path):
    hlh_data = {}

    with h5py.File(lhl_path, 'r') as f:

        stage_versions = [x.split('stage_pred_')[-1] for x in f.keys() if 'stage_pred_' in x]
        keys_h5 = list(f.keys())

        posix = [f[f'posix_{x}'] for x in stage_versions]
        h_cnn = [f[f'h_cnn_{x}'] for x in stage_versions]

        for x in stage_versions:
            hlh_data[f'h_lstm_{x}'] = f[f'h_lstm_{x}'][:]
            hlh_data[f'posix_{x}'] = f[f'posix_{x}'][:]
            hlh_data[f'yp_lstm_{x}'] = f[f'yp_lstm_{x}'][:]
            hlh_data[f'stage_pred_{x}'] = f[f'stage_pred_{x}'][:]
            hlh_data[f'study_id_{x}'] = f[f'study_id_{x}'][:]

    return hlh_data

def hlh_to_df(hlh_data, stage_versions):
    
    data = pd.DataFrame()

    for x in stage_versions:

        var = f'h_lstm_{x}'
        h_lstm_cols = [f'{var}_{i}' for i in range(hlh_data[var].shape[1])]
        data_tmp = pd.DataFrame(hlh_data[var], columns=h_lstm_cols)
        data = pd.concat([data, data_tmp], axis=1)

        var = f'posix_{x}'
        data_tmp = pd.DataFrame(hlh_data[var], columns=[var])
        data = pd.concat([data, data_tmp], axis=1)

        var = f'yp_lstm_{x}'
        yp_lstm_cols = [f'{var}_{i}' for i in range(hlh_data[var].shape[1])]
        data_tmp = pd.DataFrame(hlh_data[var], columns=yp_lstm_cols)
        data = pd.concat([data, data_tmp], axis=1)

        var = f'stage_pred_{x}'
        data_tmp = pd.DataFrame(hlh_data[var], columns=[var])
        data = pd.concat([data, data_tmp], axis=1)
        
    # only one study id (ok for overlap version)
    var = f'study_id_{x}'
    data_tmp = pd.DataFrame(hlh_data[var], columns=['study_id'])
    data = pd.concat([data, data_tmp], axis=1)        

    return data

if 0:
    assert np.all(posix[0] == posix[1])
    assert np.all(posix[0] == posix[2])
    
    

In [4]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3338: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,7880

# of subjects ICU before inclusion criteria: 102
# of 12-hour segments ICU before inclusion criteria: 1161
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [5]:
sleeplab_studyids_matched = summary_days_sleeplab.loc[summary_days_sleeplab.matched_all == 1, 'study_id'].values

In [6]:
for population in ['sleeplab', 'icu']:

    data_dir = f'/scr/wolfgang/Sleep_And_Breathing/{population}_files_sleep_staging_lhl'
    lhl_path = os.path.join(data_dir, 'staging_lhl_overlap.h5')

    print_hlh_metadata(lhl_path)
    hlh_data = load_hlh_data(lhl_path)
    stage_versions = [x.replace('stage_pred_', '') for x in hlh_data.keys() if 'stage_pred' in x]
    if 0:
        print('h_lstm shapes:')
        for x in stage_versions:
            print(f"{x:>17}: {hlh_data[f'h_lstm_{x}'].shape}")
    data = hlh_to_df(hlh_data, stage_versions)
    data['population'] = population

    if population == 'sleeplab':
        data_sleeplab = data
        del data

    elif population == 'icu':
        data_icu = data
        del data

['h_cnn_activity10sec', 'h_cnn_amendsumeffort', 'h_cnn_ecg_nn', 'h_lstm_activity10sec', 'h_lstm_amendsumeffort', 'h_lstm_ecg_nn', 'posix_activity10sec', 'posix_amendsumeffort', 'posix_ecg_nn', 'stage_pred_activity10sec', 'stage_pred_amendsumeffort', 'stage_pred_ecg_nn', 'study_id_activity10sec', 'study_id_amendsumeffort', 'study_id_ecg_nn', 'yp_cnn_activity10sec', 'yp_cnn_amendsumeffort', 'yp_cnn_ecg_nn', 'yp_lstm_activity10sec', 'yp_lstm_amendsumeffort', 'yp_lstm_ecg_nn']
['h_cnn_activity10sec', 'h_cnn_amendsumeffort', 'h_cnn_ecg_nn', 'h_lstm_activity10sec', 'h_lstm_amendsumeffort', 'h_lstm_ecg_nn', 'posix_activity10sec', 'posix_amendsumeffort', 'posix_ecg_nn', 'stage_pred_activity10sec', 'stage_pred_amendsumeffort', 'stage_pred_ecg_nn', 'study_id_activity10sec', 'study_id_amendsumeffort', 'study_id_ecg_nn', 'yp_cnn_activity10sec', 'yp_cnn_amendsumeffort', 'yp_cnn_ecg_nn', 'yp_lstm_activity10sec', 'yp_lstm_amendsumeffort', 'yp_lstm_ecg_nn']


In [7]:
lhl_path

'/scr/wolfgang/Sleep_And_Breathing/icu_files_sleep_staging_lhl/staging_lhl_overlap.h5'

In [8]:
# data_tmp.loc[data_tmp.study_id == 35, 'stage_pred_amendsumeffort']

In [9]:
print(data_sleeplab.shape)
print(data_icu.shape)
assert data_sleeplab.shape[1] == data_icu.shape[1], 'Currently, expect same data format for sleeplab and icu'

(324928, 323)
(458284, 323)


In [11]:
matching = 'ahi_all'
matching = 'all_sleeplab'

if matching == 'all_sleeplab':
    data = pd.concat([data_sleeplab, data_icu], axis = 0, ignore_index=True) 
    
if matching == 'ahi_all':
    # all ICU patients, all matched sleeplab patients:
    data = pd.concat([data_sleeplab.loc[np.isin(data_sleeplab.study_id, sleeplab_studyids_matched), :], data_icu], axis = 0, ignore_index=True) 

# TMP: reduce data by factor 10

In [12]:
if 0:
    data = data.iloc[::50, :]
    data.reset_index(inplace=True)

In [13]:
print(data.shape)
data.head(2)

(783212, 323)


,h_lstm_activity10sec_0,h_lstm_activity10sec_1,h_lstm_activity10sec_2,h_lstm_activity10sec_3,h_lstm_activity10sec_4,h_lstm_activity10sec_5,h_lstm_activity10sec_6,h_lstm_activity10sec_7,h_lstm_activity10sec_8,h_lstm_activity10sec_9,...,h_lstm_ecg_nn_39,posix_ecg_nn,yp_lstm_ecg_nn_0,yp_lstm_ecg_nn_1,yp_lstm_ecg_nn_2,yp_lstm_ecg_nn_3,yp_lstm_ecg_nn_4,stage_pred_ecg_nn,study_id,population
0,0.072801,-0.276859,-0.164528,0.125449,0.454261,0.08387,-0.187639,0.029430,-0.295639,0.207709,...,-0.352971,1.548280e+18,0.000902,0.014349,0.307574,0.000644,0.676532,5.0,1,sleeplab
1,0.017553,-0.551813,-0.385078,0.204926,0.653227,0.08124,-0.259111,0.107733,-0.554422,0.349176,...,-0.271970,1.548280e+18,0.000995,0.014102,0.232662,0.000492,0.751749,5.0,1,sleeplab


In [12]:
stages_names = ['N3', 'N2', 'N1', 'R', 'W']

In [13]:
h_lstm_ecg_columns = [x for x in data.columns if 'h_lstm_ecg_nn' in x]
h_lstm_amendsumeffort_columns = [x for x in data.columns if 'h_lstm_amendsumeffort' in x]
# variant = 'ecg_nn_lstm'
print(len(h_lstm_ecg_columns))
print(len(h_lstm_amendsumeffort_columns))

40
200


### Train PCA and UMAP

In [13]:
for variant in ['ecg_nn_lstm', 'amendsumeffort_lstm', 'ecg_and_breathing_lstm']:

    if variant == 'ecg_nn_lstm':
        stage_version = 'ecg_nn'
        features = h_lstm_ecg_columns
        label_var = f'stage_pred_{stage_version}'

    elif variant == 'amendsumeffort_lstm':
        stage_version = 'amendsumeffort'
        features = h_lstm_amendsumeffort_columns
        label_var = f'stage_pred_{stage_version}'

    elif variant == 'ecg_and_breathing_lstm':
        stage_version = None
        features = h_lstm_amendsumeffort_columns + h_lstm_ecg_columns
        label_var = None
    
    
    # PCA:
    pca = PCA()
    pca_result = pca.fit_transform(data[features].values)
    
    pca_cols = [f'pca_{variant}_{i}' for i in range(5)]
    data_tmp = pd.DataFrame(pca_result[:, :5], columns=pca_cols)
    data = pd.concat([data, data_tmp], axis=1)
    
    # UMAP:
    umap_reducer = umap.UMAP()
    umap_embedding = umap_reducer.fit_transform(data[features].values)
    data[f'umap_{variant}_2d_0'] = umap_embedding[:,0]
    data[f'umap_{variant}_2d_1'] = umap_embedding[:,1]

In [134]:
import pickle
if 0:
    pickle.dump(data, open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_pca_umap.p", "wb" ),protocol=pickle.HIGHEST_PROTOCOL)
elif 1:
    data = pickle.load( open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_pca_umap.p", "rb" ) )
elif 0:
    pickle.dump(data, open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_expert_label.p", "wb" ),protocol=pickle.HIGHEST_PROTOCOL)
elif 0:
    data = pickle.load( open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_expert_label.p", "rb" ) )
    
stages_names = ['N3', 'N2', 'N1', 'R', 'W']

### add datetime to data:

In [26]:
data['datetime'] = data['posix_amendsumeffort'].apply(lambda x: pd.to_datetime(x))

### add expert label to 'overlap' data.

In [14]:
if 0:
    dir_sleeplab = f'/scr/wolfgang/Sleep_And_Breathing/sleeplab_files_step6'
    
    for study_id in data.loc[data.population=='sleeplab', 'study_id'].unique():
        data_tmp, hdr_tmp, fs_tmp = read_in_routine(os.path.join(dir_sleeplab, f'psg_airgo_10hz_{str(study_id).zfill(3)}.h5'))
        sleeplab_study_id_loc = (data['study_id'] == study_id) & (data['population'] == 'sleeplab')
        assert np.all(np.isin(data.loc[sleeplab_study_id_loc, 'datetime'].values, data_tmp.index.values)), 'Not all timestamps in sleeplab file?'
        # copy over the stage predictions.
        datetime_hlh_file_sel = data.loc[sleeplab_study_id_loc, 'datetime'].values
        data.loc[sleeplab_study_id_loc, 'stage_pred_expert'] = data_tmp.loc[datetime_hlh_file_sel, 'stage_pred_expert'].values

In [15]:
if 0:
    
    data['ecg_nn_amendsumeffort_agreement'] = (data['stage_pred_ecg_nn'] == data['stage_pred_amendsumeffort']).astype(int)

    # strong disagreement: N3 vs. everything else than N2, N2 vs W, R vs. everything else.
    # or logical equivalent: relaxed agreement = perfect agreement or N2-N3 confusion, N1-N2 confusion, W-N1 confusion

    data['ecg_nn_amendsumeffort_agreement_relaxed'] = ( \
    (data['stage_pred_ecg_nn'] == data['stage_pred_amendsumeffort']) | \
    (np.isin(data['stage_pred_ecg_nn'], [3, 5]) & np.isin(data['stage_pred_amendsumeffort'], [3, 5])) | \
    (np.isin(data['stage_pred_ecg_nn'], [2, 3]) & np.isin(data['stage_pred_amendsumeffort'], [2, 3])) | \
    (np.isin(data['stage_pred_ecg_nn'], [1, 2]) & np.isin(data['stage_pred_amendsumeffort'], [1, 2])) )
# 1. perfect agreement, 2. W-N1 agreement, 3. N1-N2 agreement, 4. N2-N3 agreement.

In [16]:
# read in summary table so we can either fill in new values or do new stuff.

In [17]:
summary_days_icu.head()

,inclusion_subject,study_id,mrn,enrolled,age,bmi,sex,osa_diagnosis_yn,osa_diagnosis,icu_mortality,...,hours_sleep_expert_disagreement_relaxed,perc_W_expert_disagreement_relaxed,perc_S_expert_disagreement_relaxed,perc_R_expert_disagreement_relaxed,perc_N1_expert_disagreement_relaxed,perc_N2_expert_disagreement_relaxed,perc_N3_expert_disagreement_relaxed,sfi_expert_disagreement_relaxed,sfi_w_expert_disagreement_relaxed,arousali_expert_disagreement_relaxed
0,1,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,...,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,...,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,...,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,...,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,...,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Goals:
a) Show agreement between models increases agreement on sleeplab.
b) Use agreement in ICU to have a more reliable assessment of sleep statistic. With additional consequence of having 'uncertain' or 'discordant' sleep

ToDo:
1) show model performance for agreement vs. disagreement for sleeplab. Hypothesis: with agreement, agreement to expert increases as well.
2) compute sleep statistics for ICU for agreement only (hypothesis: more reliable). 
3) compute for sleeplab, icu for all models (also expert when agreement/disagreement).
   statistics per model: straightforward.
   hours certain sleep (same for ecg and breathing)
   hours uncertain sleep (same for ecg and breathing for overlap data, otherwise different for ecg and breathing)

Pseudo Algo:
1. select data: study_id, datetime, agreement/disagreement/all
    2. compute sleep indices for ecg, breathing, expert


In [18]:
data

,h_lstm_activity10sec_0,h_lstm_activity10sec_1,h_lstm_activity10sec_2,h_lstm_activity10sec_3,h_lstm_activity10sec_4,h_lstm_activity10sec_5,h_lstm_activity10sec_6,h_lstm_activity10sec_7,h_lstm_activity10sec_8,h_lstm_activity10sec_9,...,yp_lstm_ecg_nn_2,yp_lstm_ecg_nn_3,yp_lstm_ecg_nn_4,stage_pred_ecg_nn,study_id,population,datetime,stage_pred_expert,ecg_nn_amendsumeffort_agreement,ecg_nn_amendsumeffort_agreement_relaxed
0,0.072801,-0.276859,-0.164528,0.125449,0.454261,0.083870,-0.187639,0.029430,-0.295639,0.207709,...,0.307574,0.000644,0.676532,5.0,1,sleeplab,2019-01-23 21:52:59.400,5.0,1,True
1,0.017553,-0.551813,-0.385078,0.204926,0.653227,0.081240,-0.259111,0.107733,-0.554422,0.349176,...,0.232662,0.000492,0.751749,5.0,1,sleeplab,2019-01-23 21:53:29.400,5.0,1,True
2,-0.224729,-0.709088,-0.529469,0.349426,0.779519,-0.039405,-0.404612,0.221145,-0.728877,0.441562,...,0.314296,0.000515,0.664877,5.0,1,sleeplab,2019-01-23 21:53:59.400,5.0,1,True
3,-0.470771,-0.803048,-0.594357,0.484828,0.835572,-0.173096,-0.554121,0.306002,-0.789201,0.490719,...,0.321085,0.000530,0.652305,5.0,1,sleeplab,2019-01-23 21:54:29.400,5.0,1,True
4,-0.411528,-0.813928,-0.663317,0.436657,0.864693,-0.209493,-0.648437,0.433279,-0.851599,0.539771,...,0.348855,0.000510,0.623578,5.0,1,sleeplab,2019-01-23 21:54:59.400,5.0,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
783207,-0.883188,0.343511,-0.826896,-0.457192,0.634813,0.121697,-0.538378,-0.863107,-0.374181,0.771655,...,0.292259,0.123693,0.462039,5.0,189,icu,2019-11-15 04:37:30.000,NaN,0,False
783208,-0.903007,0.334598,-0.827523,-0.486238,0.658867,0.115766,-0.550866,-0.853488,-0.334456,0.736573,...,0.282292,0.128879,0.471551,5.0,189,icu,2019-11-15 04:38:00.000,NaN,0,False
783209,-0.902411,0.311461,-0.822163,-0.501657,0.665224,0.195291,-0.559573,-0.833387,-0.292968,0.728967,...,0.277548,0.126901,0.485444,5.0,189,icu,2019-11-15 04:38:30.000,NaN,0,False
783210,-0.892810,0.312367,-0.822124,-0.508752,0.681707,0.248484,-0.549605,-0.837581,-0.301903,0.751477,...,0.284235,0.125236,0.484907,5.0,189,icu,2019-11-15 04:39:00.000,NaN,0,False


In [22]:
population = 'icu'
study_id_tmp = 11

data.loc[(data.population==population) & (data.study_id==study_id_tmp)]



,h_lstm_activity10sec_0,h_lstm_activity10sec_1,h_lstm_activity10sec_2,h_lstm_activity10sec_3,h_lstm_activity10sec_4,h_lstm_activity10sec_5,h_lstm_activity10sec_6,h_lstm_activity10sec_7,h_lstm_activity10sec_8,h_lstm_activity10sec_9,...,h_lstm_ecg_nn_39,posix_ecg_nn,yp_lstm_ecg_nn_0,yp_lstm_ecg_nn_1,yp_lstm_ecg_nn_2,yp_lstm_ecg_nn_3,yp_lstm_ecg_nn_4,stage_pred_ecg_nn,study_id,population


In [19]:
population = 'sleeplab'

for population in ['sleeplab', 'icu']:

    if population == 'icu':
        summary_days = summary_days_icu.copy()
    elif population == 'sleeplab':
        summary_days = summary_days_sleeplab.copy()


    fs = 1/30 # hlh data in 30 seconds epoch.
    stages = ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort', 'stage_pred_expert']
    agreements = ['all', 'agreement', 'disagreement', 'agreement_relaxed', 'disagreement_relaxed']


    for jloc, row in summary_days.iterrows():
        statistics_result = pd.DataFrame()

        start_dt = row.timerange.split(' - ')[0]
        end_dt = row.timerange.split(' - ')[1]

        data_subject = data.loc[(data.population==population) & (data.study_id==row.study_id)]
        data_timerange = data_subject.loc[(data_subject.datetime >= start_dt) & (data_subject.datetime <= end_dt)]


        for stage, agreement in itertools.product(stages, agreements):
            stage_short = stage.replace('stage_pred_', '')
            name = stage_short + '_' + agreement

            if agreement == 'all':
                data_sel = data_timerange
            elif agreement == 'agreement':
                data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement == 1]
            elif agreement == 'disagreement':
                data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement == 0]
            elif agreement == 'agreement_relaxed':
                data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement_relaxed == 1]
            elif agreement == 'disagreement_relaxed':
                data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement_relaxed == 0]

            statistics_result_tmp = compute_sleep_indices(data_sel, stage, fs, name=name)
            statistics_result = pd.concat([statistics_result, statistics_result_tmp], axis=1)

            statistics_result_tmp.index += '_' + name

            for new_col in list(statistics_result_tmp.index):
                if not new_col in summary_days.columns:
                    summary_days[new_col] = np.nan

            summary_days.loc[jloc, list(statistics_result_tmp.index)] = statistics_result_tmp.values.flatten()

    if population == 'icu':
        summary_days_icu = summary_days.copy()
    elif population == 'sleeplab':
        summary_days_sleeplab = summary_days.copy()

In [20]:
summary_days_sleeplab.iloc[:2,-150:]

,perc_R_ecg_nn_agreement,perc_N1_ecg_nn_agreement,perc_N2_ecg_nn_agreement,perc_N3_ecg_nn_agreement,sfi_ecg_nn_agreement,sfi_w_ecg_nn_agreement,arousali_ecg_nn_agreement,hours_data_ecg_nn_disagreement,hours_sleep_ecg_nn_disagreement,perc_W_ecg_nn_disagreement,...,hours_sleep_expert_disagreement_relaxed,perc_W_expert_disagreement_relaxed,perc_S_expert_disagreement_relaxed,perc_R_expert_disagreement_relaxed,perc_N1_expert_disagreement_relaxed,perc_N2_expert_disagreement_relaxed,perc_N3_expert_disagreement_relaxed,sfi_expert_disagreement_relaxed,sfi_w_expert_disagreement_relaxed,arousali_expert_disagreement_relaxed
0,0.471111,0.093333,0.435555,0.000000,1.6,0.5,0.5,4.050001,3.816668,0.057613,...,1.416668,0.055556,0.944444,0.229412,0.017647,0.529411,0.223529,4.9,4.2,4.2
1,0.177388,0.066277,0.569201,0.187134,2.3,0.5,0.9,1.516668,1.450001,0.043956,...,0.333334,0.000000,0.999997,0.474999,0.075000,0.424999,0.025000,6.0,0.0,0.0


In [22]:
if 0:
    summary_days_icu.to_csv('summary_days_icu.csv', index=False)
    summary_days_sleeplab.to_csv('summary_days_sleeplab.csv', index=False)

In [62]:
data.head(2)

,h_lstm_activity10sec_0,h_lstm_activity10sec_1,h_lstm_activity10sec_2,h_lstm_activity10sec_3,h_lstm_activity10sec_4,h_lstm_activity10sec_5,h_lstm_activity10sec_6,h_lstm_activity10sec_7,h_lstm_activity10sec_8,h_lstm_activity10sec_9,...,yp_lstm_ecg_nn_2,yp_lstm_ecg_nn_3,yp_lstm_ecg_nn_4,stage_pred_ecg_nn,study_id,population,datetime,stage_pred_expert,ecg_nn_amendsumeffort_agreement,ecg_nn_amendsumeffort_agreement_relaxed
0,0.072801,-0.276859,-0.164528,0.125449,0.454261,0.08387,-0.187639,0.029430,-0.295639,0.207709,...,0.307574,0.000644,0.676532,5.0,1,sleeplab,2019-01-23 21:52:59.400,5.0,1,True
1,0.017553,-0.551813,-0.385078,0.204926,0.653227,0.08124,-0.259111,0.107733,-0.554422,0.349176,...,0.232662,0.000492,0.751749,5.0,1,sleeplab,2019-01-23 21:53:29.400,5.0,1,True


In [49]:
def select_data_plot(data, population=None, hue=None, hue_order=None):
    
    if population is None: 
        data_plot = None
    else:
        if population == 'all':
            data_plot = data.copy()
        elif population == 'icu':
            data_plot = data.loc[data.population=='icu', :].copy()
        elif population == 'sleeplab':
            data_plot = data.loc[data.population=='sleeplab', :].copy()

    if hue is not None:
        if hue in ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']:
            hue = stages_to_names(data_plot[hue])
            hue_order = stages_names
            
        elif hue == 'stage_pred_ecg_nn_population':
            hue = "stage_pred_ecg_nn"
            hue = stages_to_names(data_plot[hue])
            hue = hue + '_' + data_plot.population.values
            hue_order = [x + '_'+ y for x in stages_names for y in ['sleeplab', 'icu']]
        elif hue == 'stage_pred_amendsumeffort_population':
            hue = "stage_pred_amendsumeffort"
            hue = stages_to_names(data_plot[hue])
            hue = hue + '_' + data_plot.population.values
            hue_order = [x + '_'+ y for x in stages_names for y in ['sleeplab', 'icu']]
            
        elif hue in [f'amendsumeffort_{x}_population' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
                    [f'ecg_nn_{x}_population' for x in ['N1', 'N2', 'N3', 'W', 'R']]:
            stage = hue.split('_')[-2]
            
            if stage == 'N1': stage_no = 3
            elif stage == 'N2': stage_no = 2
            elif stage == 'N3': stage_no = 1
            elif stage == 'R': stage_no = 4
            elif stage == 'W': stage_no = 5
            else: raise ValueError('unclear sleep stage.')
            
            model_version = hue.split(f'_{stage}_')[0]
            data_plot = data_plot.loc[data_plot['stage_pred_' + model_version] == stage_no]
            
            hue = stages_to_names(data_plot['stage_pred_' + model_version])
            hue = hue + '_' + data_plot.population.values
            hue_order = [stage + '_'+ y for y in ['sleeplab', 'icu']]

        elif hue in [f'amendsumeffort_{x}_agreementrelaxed_population' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
                    [f'ecg_nn_{x}_agreementrelaxed_population' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
        [f'amendsumeffort_{x}_agreementrelaxed_sleeplab' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
                    [f'ecg_nn_{x}_agreementrelaxed_sleeplab' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
        [f'amendsumeffort_{x}_agreementrelaxed_icu' for x in ['N1', 'N2', 'N3', 'W', 'R']] + \
                    [f'ecg_nn_{x}_agreementrelaxed_icu' for x in ['N1', 'N2', 'N3', 'W', 'R']]:
            
            stage = hue.split('_')[-3]
            
            if stage == 'N1': stage_no = 3
            elif stage == 'N2': stage_no = 2
            elif stage == 'N3': stage_no = 1
            elif stage == 'R': stage_no = 4
            elif stage == 'W': stage_no = 5
            else: raise ValueError('unclear sleep stage.')
                
            model_version = hue.split(f'_{stage}_')[0]
            data_plot = data_plot.loc[data_plot['stage_pred_' + model_version] == stage_no]
                        
            hue = data_plot['ecg_nn_amendsumeffort_agreement_relaxed']
            hue[hue == 0] = 'Disagreement'
            hue[hue == 1] = 'Agreement'
            if 'population' in hue:
                hue = data_plot.population.values + ' ' + hue
                hue_order = ['sleeplab Agreement', 'sleeplab Disagreement', 'icu Agreement', 'icu Disagreement']    
            else:
                hue_order = ['Agreement', 'Disagreement']  
                    
    return data_plot, hue, hue_order


def u_map_plot(ax, data, hue, variant, hue_order=None, alpha=0.3, s=2.5, legend='full', plot_type ='scatter',
              bw_adjust=0.8, levels=np.arange(0.1, 1, 0.2), fill=False):

    hue_no = len(np.unique(hue))
    if hue_no == 2:  
        
        if 'Agreement' in hue_order:
            cmap = [
                (34/255, 139/255, 34/255),
                (1, 69/255, 0)]
        else: # icu vs sleeplab plots
            cmap = [
                (0.86, 0.65, 0.13),
                (0, 0, 128/255)]
    elif hue_no == 4: # agree, disagreement plots
        cmap = [
            (173/255, 1, 50/255),
            (1, 0, 0),
            (32/255, 178/255, 170/255),
            (1, 165/255, 0),
        ]
    elif hue_no == 5: # 5 stages
        cmap = [
            (220/255, 20/255, 60/255),
            (1, 165/255, 0),
            (173/255, 1, 47/255),
            (0, 0, 1),
            (0, 0 , 0),
        ]
    elif hue_no == 10: # 5 stages, 2 populations
        cmap = [
            (220/255, 20/255, 60/255),   #  N3
            (255/255, 20/255, 147/255),
            (1, 165/255, 0),             #  N2
            (1, 255/255, 0),
            (173/255, 1, 47/255),        #  N1
            (0/255, 100/255, 0/255),
            (0, 0, 1),                   # R
            (0, 191/255, 1),
            (0.5, 0.5 , 0.5),                  # W
            (0, 0 , 0),                
        ]
        
    else:
        raise ValueError(f'Unexpected number of Hues: {np.unique(hue)}')
    
    if plot_type == 'scatter':
        sns.scatterplot(
            x=f'umap_{variant}_2d_0', y=f'umap_{variant}_2d_1',
            hue=hue,
            hue_order = hue_order[::-1],
            palette=cmap[::-1],
            data=data,
            legend=legend,
            alpha=alpha,
            s=s,
            ax=ax)
        
    elif plot_type == 'kde':
        sns.kdeplot(
            x=f'umap_{variant}_2d_0', y=f'umap_{variant}_2d_1',
            hue=hue,
            hue_order = hue_order[::-1],
            palette=cmap[::-1],
            data=data,
            legend=legend,
            alpha = alpha,
            thresh=0.05,
            levels=levels,
            bw_adjust=bw_adjust,
            fill = fill,
            ax=ax)

    return ax

### Principal Component Analysis

In [15]:
pca = PCA()

In [16]:
pca_result = pca.fit_transform(data[features].values)

In [17]:
print('Cumulative Variance Explained by Principal Components:')
print(f'Components 1-2 contain {pca.explained_variance_ratio_.cumsum()[1]*100:.2f}% of the variance')
print(f'Components 1-3 contain {pca.explained_variance_ratio_.cumsum()[2]*100:.2f}% of the variance')
print(f'Components 1-5 contain {pca.explained_variance_ratio_.cumsum()[4]*100:.2f}% of the variance')
# print(pca.explained_variance_ratio_.cumsum())

Cumulative Variance Explained by Principal Components:
Components 1-2 contain 55.74% of the variance
Components 1-3 contain 65.14% of the variance
Components 1-5 contain 74.19% of the variance


In [18]:
pca_cols = [f'pca_{variant}_{i}' for i in range(5)]
data_tmp = pd.DataFrame(pca_result[:, :5], columns=pca_cols)
data = pd.concat([data, data_tmp], axis=1)

In [19]:
data.head(2)

,index,h_lstm_activity10sec_0,h_lstm_activity10sec_1,h_lstm_activity10sec_2,h_lstm_activity10sec_3,h_lstm_activity10sec_4,h_lstm_activity10sec_5,h_lstm_activity10sec_6,h_lstm_activity10sec_7,h_lstm_activity10sec_8,...,yp_lstm_ecg_nn_3,yp_lstm_ecg_nn_4,stage_pred_ecg_nn,study_id,population,pca_ecg_nn_lstm_0,pca_ecg_nn_lstm_1,pca_ecg_nn_lstm_2,pca_ecg_nn_lstm_3,pca_ecg_nn_lstm_4
0,0,0.13519,-0.313588,0.133864,-0.174114,0.410875,-0.305102,-0.402843,0.062565,0.022724,...,0.193473,0.312684,5.0,6,sleeplab,0.412598,0.387162,-0.363337,-0.116145,-0.443689
1,10,0.40900,-0.918113,-0.447826,-0.789204,0.849431,-0.680278,-0.910078,0.265248,-0.028324,...,0.114517,0.326796,3.0,6,sleeplab,0.387918,0.449303,-0.177822,-0.707366,0.148952


In [256]:
data.shape

(543194, 346)

In [257]:
stage_version = 'stage_pred_ecg_nn'

In [261]:
stages_names

['N3', 'N2', 'N1', 'R', 'W']

In [263]:
pca_cols

['pca_ecg_and_breathing_lstm_0',
 'pca_ecg_and_breathing_lstm_1',
 'pca_ecg_and_breathing_lstm_2',
 'pca_ecg_and_breathing_lstm_3',
 'pca_ecg_and_breathing_lstm_4']

In [114]:
plt.close('all')

In [26]:
cmap = [
    (220/255, 20/255, 60/255),
    (1, 165/255, 0),
    (173/255, 1, 47/255),
    (0, 0, 1),
    (0, 0 , 0),
]

population_plot = 'sleeplab'
population_plot = 'icu'
population_plot = 'all'

# hue_version = 'stage_pred_ecg_nn_population'

for population_plot in ['all', 'sleeplab', 'icu']:

    s=4
    alpha=0.025
    text_x_position = -0.4

    fig, ax = plt.subplots(2,3, figsize=(7,3.5))

    variant = 'ecg_nn_lstm'
    hue_version = 'stage_pred_ecg_nn'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    hue = stages_to_names(data[hue_version])

    i = 0
    j = 0
    sns.scatterplot(
        x='pca_' + variant + '_0', 
        y = 'pca_' + variant + '_1',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend=False,
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 1')
    ax[i, j].set_ylabel(f'PC 2')
    ax[i, j].text(text_x_position, 0.5, 'HRV', color='black', rotation=90, 
                  horizontalalignment='center', verticalalignment='center', transform=ax[i, j].transAxes)

    j = 1
    sns.scatterplot(
        x='pca_' + variant + '_0', 
        y = 'pca_' + variant + '_2',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend=False,
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 1')
    ax[i, j].set_ylabel(f'PC 3')

    j = 2
    ax[i, j] = sns.scatterplot(
        x='pca_' + variant + '_1', 
        y = 'pca_' + variant + '_2',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend='full',
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 2')
    ax[i, j].set_ylabel(f'PC 3')
    ax[i, j].legend(bbox_to_anchor=(1.01, 1),borderaxespad=0, frameon=False)

    variant = 'amendsumeffort_lstm'
    hue_version = 'stage_pred_amendsumeffort'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    hue = stages_to_names(data[hue_version])

    i = 1
    j = 0
    sns.scatterplot(
        x='pca_' + variant + '_0', 
        y = 'pca_' + variant + '_1',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend=False,
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 1')
    ax[i, j].set_ylabel(f'PC 2')
    ax[i, j].text(text_x_position, 0.5, 'Breathing', color='black', rotation=90, 
                  horizontalalignment='center', verticalalignment='center', transform=ax[i, j].transAxes)


    j = 1
    sns.scatterplot(
        x='pca_' + variant + '_0', 
        y = 'pca_' + variant + '_2',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend=False,
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 1')
    ax[i, j].set_ylabel(f'PC 3')

    j = 2
    ax[i, j] = sns.scatterplot(
        x='pca_' + variant + '_1', 
        y = 'pca_' + variant + '_2',
        hue=hue,
        hue_order = stages_names[::-1],
        palette=cmap[::-1],
        data=data_plot,
        s=s,
        alpha=alpha,
        legend=False,
        ax = ax[i, j],
    )
    ax[i, j].set_xlabel(f'PC 2')
    ax[i, j].set_ylabel(f'PC 3')

    c_axis = 'dimgray'
    ticklength = 1
    
    for i in range(2):
        for j in range(3):
            ax[i, j].spines['bottom'].set_color(c_axis)
            ax[i, j].spines['top'].set_color(c_axis)
            ax[i, j].spines['right'].set_color(c_axis)
            ax[i, j].spines['left'].set_color(c_axis)
            ax[i, j].tick_params(axis='x', colors=c_axis, length=ticklength)
            ax[i, j].tick_params(axis='y', colors=c_axis, length=ticklength)
            ax[i, j].yaxis.label.set_color(c_axis)
            ax[i, j].xaxis.label.set_color(c_axis)

    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    plt.tight_layout()
    # plt.savefig(f'PCA_{population_plot}.svg')
    plt.savefig(f'PCA_{population_plot}.jpg', dpi=500)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [300]:
plt.close('all')

In [28]:
if 0:
    
    plt.figure(figsize=(8,4))
    g = sns.scatterplot(
        x=pca_cols[1], y=pca_cols[2],
        hue=hue,
        hue_order = stages_names,
        palette=sns.color_palette("hls", 5),
        data=data,
        legend="full",
        alpha=0.3
    )
    plt.tight_layout()

In [27]:
if 0:
    
    hue = "stage_pred_ecg_nn"
    hue = data[hue]

    ax = plt.figure(figsize=(8,6)).gca(projection='3d')

    ax.scatter(
        xs=data.iloc[::100]["pca_0"], 
        ys=data.iloc[::100]["pca_1"], 
        zs=data.iloc[::100]["pca_2"], 
        c=hue[::100], 
        cmap='tab10'
    )
    ax.set_xlabel('pca-one')
    ax.set_ylabel('pca-two')
    ax.set_zlabel('pca-three')
    plt.show()

### t-SNE

In [25]:
if 0: 
    
    time_start = time.time()
    n_jobs = -1
    tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300, n_jobs = n_jobs)
    tsne_results = tsne.fit_transform(data[features].values)
    print('t-SNE done! Time elapsed: {} seconds'.format(time.time()-time_start))
    
    
    hue = "stage_pred_ecg_nn"
    hue = stages_to_names(data[hue])

    data['tsne_2d_0'] = tsne_results[:,0]
    data['tsne_2d_1'] = tsne_results[:,1]
    plt.figure(figsize=(10,7))
    sns.scatterplot(
        x="tsne_2d_0", y="tsne_2d_1",
        hue=hue,
        hue_order = stages_names,
        palette=sns.color_palette("hls", 5),
        data=data,
        legend="full",
        alpha=0.3,
        s = 2.5
    )


### UMAP

In [26]:
time_start = time.time()
umap_reducer = umap.UMAP()
umap_embedding = umap_reducer.fit_transform(data[features].values)
print('UMAP done! Time elapsed: {} seconds'.format(time.time()-time_start))

UMAP done! Time elapsed: 36.92946648597717 seconds


In [27]:
print(umap_embedding.shape)

(54320, 2)


In [28]:
data[f'umap_{variant}_2d_0'] = umap_embedding[:,0]
data[f'umap_{variant}_2d_1'] = umap_embedding[:,1]

In [188]:
variant

'ecg_nn_lstm'

In [189]:
data.columns[-22:]

Index(['pca_ecg_nn_lstm_1', 'pca_ecg_nn_lstm_2', 'pca_ecg_nn_lstm_3',
       'pca_ecg_nn_lstm_4', 'umap_ecg_nn_lstm_2d_0', 'umap_ecg_nn_lstm_2d_1',
       'pca_amendsumeffort_lstm_0', 'pca_amendsumeffort_lstm_1',
       'pca_amendsumeffort_lstm_2', 'pca_amendsumeffort_lstm_3',
       'pca_amendsumeffort_lstm_4', 'umap_amendsumeffort_lstm_2d_0',
       'umap_amendsumeffort_lstm_2d_1', 'pca_ecg_and_breathing_lstm_0',
       'pca_ecg_and_breathing_lstm_1', 'pca_ecg_and_breathing_lstm_2',
       'pca_ecg_and_breathing_lstm_3', 'pca_ecg_and_breathing_lstm_4',
       'umap_ecg_and_breathing_lstm_2d_0', 'umap_ecg_and_breathing_lstm_2d_1',
       'ecg_nn_amendsumeffort_agreement',
       'ecg_nn_amendsumeffort_agreement_relaxed'],
      dtype='object')

In [40]:
population_plot = 'sleeplab'
population_plot = 'all'
alpha = 0.02
s = 6

if 1:
    variant = 'ecg_nn_lstm'
    hue_version = 'stage_pred_ecg_nn'
else:
    variant = 'amendsumeffort_lstm'
    hue_version = 'stage_pred_amendsumeffort'
    
for population_plot in ['all', 'sleeplab', 'icu']:

    fig, ax = plt.subplots(1, 1, figsize=(4,4))

    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax = u_map_plot(ax, data_plot, hue, variant, hue_order, alpha=alpha, s=s)
    ax.legend(title='', frameon=False)
    ax.set_xlabel('UMAP Component 1')
    ax.set_ylabel('UMAP Component 2')

    c_axis = 'dimgray'
    ticklength = 1
    ax.spines['bottom'].set_color(c_axis)
    ax.spines['top'].set_color(c_axis) 
    ax.spines['right'].set_color(c_axis)
    ax.spines['left'].set_color(c_axis)
    ax.tick_params(axis='x', colors=c_axis, length=ticklength)
    ax.tick_params(axis='y', colors=c_axis, length=ticklength)
    ax.yaxis.label.set_color(c_axis)
    ax.xaxis.label.set_color(c_axis)
    
    plt.tight_layout()
    plt.savefig(f'umap_{population_plot}_{hue_version}.jpg', dpi=700)
    

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [44]:
alpha = 0.02
s = 6

fig, ax = plt.subplots(1, 3, figsize=(10,5), sharex=True, sharey=True)
ax = ax.flatten()

variant = 'ecg_nn_lstm'
hue_version = 'stage_pred_ecg_nn'

population_plot = 'sleeplab'
i = 0
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)

i = 1
population_plot = 'icu'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].legend([])

i = 2
population_plot = 'all'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].legend([])

plt.tight_layout()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [45]:

# hue_version = 'stage_pred_ecg_nn_population'

alpha = 0.02
s = 6

fig, ax = plt.subplots(1, 3, figsize=(10,5), sharex=True, sharey=True)
ax = ax.flatten()

variant = 'amendsumeffort_lstm'
hue_version = 'stage_pred_amendsumeffort'

population_plot = 'sleeplab'
i = 0
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)

i = 1
population_plot = 'icu'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].legend([])

i = 2
population_plot = 'all'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].legend([])


plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [39]:
plt.close('all')

In [59]:
alpha = 0.02
s = 6
population_plot = 'all'

fig, ax = plt.subplots(1, 2, figsize=(10,5))

i = 0
variant = 'ecg_nn_lstm'
hue_version = 'stage_pred_ecg_nn_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV')

i = 1
variant = 'amendsumeffort_lstm'
hue_version = 'stage_pred_amendsumeffort_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing')
ax[i].legend([])

plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [115]:
# hue_version = 'stage_pred_ecg_nn_population'
population_plot = 'sleeplab'
population_plot = 'all'
alpha = 0.007
s = 4
text_x_position = -0.15

fig, ax = plt.subplots(2, 3, figsize=(7, 3.5), sharex='row', sharey='row', constrained_layout=True)
ax = ax.flatten()

variant = 'ecg_nn_lstm'
hue_version = 'stage_pred_ecg_nn'
population_plot = 'sleeplab'
i = 0
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False)
ax[i].text(0.5, 1.05, 'Sleeplab', color='black',
                  horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
ax[i].text(text_x_position, 0.5, 'HRV', color='black', rotation=90, 
                  horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
ax[i].set_xlabel('')
ax[i].set_ylabel('UMAP C2', labelpad=0)
i = 1
population_plot = 'icu'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False)
ax[i].text(0.5, 1.05, 'ICU', color='black',
                  horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
ax[i].set_xlabel('')
ax[i].set_ylabel('')
i = 2
population_plot = 'all'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend='full')
ax[i].text(0.5, 1.05, 'All', color='black',
                  horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
ax[i].legend(bbox_to_anchor=(1.01, 1), borderaxespad=0, frameon=False, handletextpad=0)
ax[i].set_xlabel('')
ax[i].set_ylabel('')
variant = 'amendsumeffort_lstm'
hue_version = 'stage_pred_amendsumeffort'
population_plot = 'sleeplab'
i = 3
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False)
ax[i].text(text_x_position, 0.5, 'Breathing', color='black', rotation=90, 
                  horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
ax[i].set_xlabel('UMAP C1', labelpad=0)
ax[i].set_ylabel('UMAP C2', labelpad=0)
i = 4
population_plot = 'icu'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False)
ax[i].set_xlabel('UMAP C1', labelpad=0)
ax[i].set_ylabel('')
i = 5
population_plot = 'all'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False)
ax[i].set_xlabel('UMAP C1', labelpad=0)
ax[i].set_ylabel('')

for i in range(6):
    c_axis = 'dimgray'
    ticklength = 0
    ax[i].spines['bottom'].set_color(c_axis)
    ax[i].spines['top'].set_color(c_axis) 
    ax[i].spines['right'].set_color(c_axis)
    ax[i].spines['left'].set_color(c_axis)
    ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
    ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
    ax[i].yaxis.label.set_color(c_axis)
    ax[i].xaxis.label.set_color(c_axis)
    
    ax[i].set_yticklabels([])
    ax[i].set_xticklabels([])
    
plt.subplots_adjust(hspace=0, wspace=0, bottom=0.045, top=0.94, left=0.06, right=0.93)
plt.text(0, 0.97, 'A', fontweight='bold', transform=fig.transFigure)
# plt.tight_layout()
plt.savefig('UMAP_population_signal_grid_A.jpg', dpi=700)
# plt.savefig('UMAP_population_signal_grid.png', dpi=700)
# plt.savefig('UMAP_population_signal_grid.svg') # way too large filesize!
plt.savefig('UMAP_population_signal_grid_A.pdf')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:78: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


### stage-wise plots

In [120]:
if 1:
    data = data.iloc[::50, :]
    data.reset_index(inplace=True)

In [65]:
for fill in [True, False]:
    s = 6
    population_plot = 'all'
    text_x_position = -0.115
    labelpad = 0
    plot_type = 'kde'
    bw_adjust = 0.8
    # fill = True

    levels = np.arange(0.1, 1.1, 0.1) # [0.05, 0.1, 0.3, 0.5, 0.75, 1]

    if plot_type == 'scatter':
        alpha = 0.01
    elif plot_type == 'kde':
        if fill:
            alpha = 0.5
        else:
            alpha = 0.6

    fig, ax = plt.subplots(5, 2, figsize=(5,7), constrained_layout=True)
    ax = ax.flatten()

    i = 8
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N3_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('UMAP C2', labelpad=labelpad)
    ax[i].text(text_x_position, 0.5, 'N3', color='black', rotation=90, 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 9
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N3_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 6
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N2_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('UMAP C2', labelpad=labelpad)
    ax[i].text(text_x_position, 0.5, 'N2', color='black', rotation=90, 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 7
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N2_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 4
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N1_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('UMAP C2', labelpad=labelpad)
    ax[i].text(text_x_position, 0.5, 'N1', color='black', rotation=90, 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 5
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N1_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 2
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_R_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('UMAP C2', labelpad=labelpad)
    ax[i].text(text_x_position, 0.5, 'R', color='black', rotation=90, 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 3
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_R_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 0
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_W_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend='full', plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_ylabel('UMAP C2', labelpad=labelpad)
    ax[i].text(text_x_position, 0.5, 'W', color='black', rotation=90, 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 1
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_W_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_ylabel('')

    ax[-1].set_xlabel('UMAP C1', labelpad=labelpad)
    ax[-2].set_xlabel('UMAP C1', labelpad=labelpad)

    ax[0].text(0.5, 1.1, 'HRV', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[0].transAxes)

    ax[1].text(0.5, 1.1, 'Breathing', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[1].transAxes)

    if plot_type == 'kde':
        colors = [
            (0.86, 0.65, 0.13),
            (0, 0, 128/255)]
        lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
        labels = ['Sleeplab', 'ICU']
        ax[0].legend(lines, labels, 
                     frameon=False, markerscale=1, ncol=2, bbox_to_anchor=(0.1, 1.05, 0.5, 0.5), loc='center')
    elif plot_type == 'scatter':
        ax[0].legend(labels=['', 'ICU', 'Sleeplab'], bbox_to_anchor=(0.65, 1.4), borderaxespad=0, frameon=False, ncol=3)

    for i in range(10):
        c_axis = 'dimgray'
        ticklength = 0
        ax[i].spines['bottom'].set_color(c_axis)
        ax[i].spines['top'].set_color(c_axis) 
        ax[i].spines['right'].set_color(c_axis)
        ax[i].spines['left'].set_color(c_axis)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

        ax[i].set_xticklabels([])
        ax[i].set_yticklabels([])

    plt.subplots_adjust(hspace=0, wspace=0, bottom=0.025, top=0.925, left=0.07, right=0.99)
    # plt.tight_layout()

    plt.savefig(f'UMAP_stage_signal_grid_{plot_type}_fill{fill}.jpg', dpi=700)
    plt.savefig(f'UMAP_stage_signal_grid_{plot_type}_fill{fill}.png', dpi=700)
    plt.savefig(f'UMAP_stage_signal_grid_{plot_type}_fill{fill}.svg')
    plt.savefig(f'UMAP_stage_signal_grid_{plot_type}_fill{fill}.pdf')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:141: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:141: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


In [135]:
for fill in [False, True]:
    s = 6
    population_plot = 'all'
    text_x_position = 0.5
    text_y_position = 1.06
    labelpad = 0
    plot_type = 'kde'
    bw_adjust = 0.8
    # fill = True

    levels = np.arange(0.1, 1.1, 0.1) # [0.05, 0.1, 0.3, 0.5, 0.75, 1]

    if plot_type == 'scatter':
        alpha = 0.01
    elif plot_type == 'kde':
        if fill:
            alpha = 0.5
        else:
            alpha = 0.6

    fig, ax = plt.subplots(2, 5, figsize=(7, 3), constrained_layout=True)
    ax = ax.flatten()

    i = 4
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N3_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].text(text_x_position, text_y_position, 'N3', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 9
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N3_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 3
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N2_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].text(text_x_position, text_y_position, 'N2', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 8
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N2_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 2
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N1_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].text(text_x_position, text_y_position, 'N1', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 7
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N1_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 1
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_R_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    ax[i].text(text_x_position, text_y_position, 'R', color='black',
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 6
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_R_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

    i = 0
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_W_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend='full', plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].text(text_x_position, text_y_position, 'W', color='black', 
                      horizontalalignment='center', verticalalignment='center', transform=ax[i].transAxes)
    i = 5
    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_W_population'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, legend=False, plot_type=plot_type, bw_adjust=bw_adjust, levels=levels, fill=fill)
    ax[i].set_ylabel('')

    ax[-1].set_xlabel('UMAP C1', labelpad=labelpad)
    ax[-2].set_xlabel('UMAP C1', labelpad=labelpad)
    ax[-3].set_xlabel('UMAP C1', labelpad=labelpad)
    ax[-4].set_xlabel('UMAP C1', labelpad=labelpad)
    ax[-5].set_xlabel('UMAP C1', labelpad=labelpad)

    ax[0].text(-0.22, 0.5, 'HRV', color='black', 
                      horizontalalignment='center', verticalalignment='center', rotation=90, transform=ax[0].transAxes)
    ax[0].set_ylabel('UMAP C2', labelpad=labelpad)

    ax[5].text(-0.22, 0.5, 'Breathing', color='black', 
                      horizontalalignment='center', verticalalignment='center', rotation=90, transform=ax[5].transAxes)
    ax[5].set_ylabel('UMAP C2', labelpad=labelpad)

    if plot_type == 'kde':
        colors = [
            (0.86, 0.65, 0.13),
            (0, 0, 128/255)]
        lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
        labels = ['Sleeplab', 'ICU']
        ax[0].legend(lines, labels, 
                     frameon=False, markerscale=1, ncol=2, bbox_to_anchor=(0.6, 1.2), loc='center')
    elif plot_type == 'scatter':
        ax[0].legend(labels=['', 'ICU', 'Sleeplab'], bbox_to_anchor=(0.5, 0.5), borderaxespad=0, frameon=False, ncol=3)

    for i in range(10):
        c_axis = 'dimgray'
        ticklength = 0
        ax[i].spines['bottom'].set_color(c_axis)
        ax[i].spines['top'].set_color(c_axis) 
        ax[i].spines['right'].set_color(c_axis)
        ax[i].spines['left'].set_color(c_axis)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

        ax[i].set_xticklabels([])
        ax[i].set_yticklabels([])

    plt.subplots_adjust(hspace=0, wspace=0, bottom=0.06, top=0.89, left=0.06, right=0.995)
    plt.text(0, 0.97, 'B', fontweight='bold', transform=fig.transFigure)
    # plt.tight_layout()

    plt.savefig(f'UMAP_stage_signal_grid_T_{plot_type}_fill{fill}.jpg', dpi=700)
    plt.savefig(f'UMAP_stage_signal_grid_T_{plot_type}_fill{fill}.png', dpi=700)
    plt.savefig(f'UMAP_stage_signal_grid_T_{plot_type}_fill{fill}.svg')
    plt.savefig(f'UMAP_stage_signal_grid_T_{plot_type}_fill{fill}.pdf')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:147: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:147: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


In [28]:
plt.show()

In [119]:
plt.close('all')

In [32]:
alpha = 0.5
s = 6
population_plot = 'all'
plot_type = 'kde'

fig, ax = plt.subplots(5, 2, figsize=(7,10))
ax = ax.flatten()

i = 0
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N3_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title('HRV-N3')

i = 1
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N3_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N3')
ax[i].legend([])

i = 2
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N2_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title('HRV-N2')
ax[i].legend([])

i = 3
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N2_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N2')
ax[i].legend([])

i = 4
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N1_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title('HRV-N1')
ax[i].legend([])

i = 5
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N1_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N1')
ax[i].legend([])

i = 6
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_R_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title('HRV-R')
ax[i].legend([])

i = 7
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_R_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-R')
ax[i].legend([])

i = 8
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_W_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title('HRV-Wake')
ax[i].legend([])

i = 9
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_W_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type='kde')
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-Wake')
ax[i].legend([])

plt.tight_layout()

# plt.savefig('stagewise_kde_plots_hrv_breathing_2.svg')
# plt.savefig('stagewise_kde_plots_hrv_breathing_2.jpg', dpi=500)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [102]:
plt.close('all')

### Agreement between ECG and breathing model.

1. agree sleeplab, disagree sleeplab, agree icu, disagree icu  
1a) % agreement | stage

'N2'

In [247]:
print('Exact stage agreement:')
print(f"% agreement all data sleeplab: {data.loc[data.population == 'sleeplab', 'ecg_nn_amendsumeffort_agreement'].mean().round(3)}")
print(f"% agreement all data icu     : {data.loc[data.population == 'icu', 'ecg_nn_amendsumeffort_agreement'].mean().round(3)}")

print(f"% Sleep Only: agreement all data sleeplab: {data.loc[(data.population == 'sleeplab') & (data.stage_pred_ecg_nn < 5), 'ecg_nn_amendsumeffort_agreement'].mean().round(3)}")
print(f"% Sleep Only: agreement all data icu     : {data.loc[(data.population == 'icu') & (data.stage_pred_ecg_nn < 5), 'ecg_nn_amendsumeffort_agreement'].mean().round(3)}")


for population_sel in ['sleeplab', 'icu']:
    for stage_pred_version in ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']:
        print(f'\nAgreement of all epochs classified as a certain sleepstage for model {stage_pred_version}')
        for stage_no in range(1,6):
            print(f"{stages_to_names(pd.Series([stage_no]))[0]:>2}: {data.loc[(data.population == population_sel) & (data[stage_pred_version] == stage_no), 'ecg_nn_amendsumeffort_agreement'].mean().round(3)}")


print('\nRelaxed stage agreement:')
print(f"% agreement all data sleeplab: {data.loc[data.population == 'sleeplab', 'ecg_nn_amendsumeffort_agreement_relaxed'].mean().round(3)}")
print(f"% agreement all data icu     : {data.loc[data.population == 'icu', 'ecg_nn_amendsumeffort_agreement_relaxed'].mean().round(3)}")
print(f"% Sleep Only: agreement all data sleeplab: {data.loc[(data.population == 'sleeplab') & (data.stage_pred_ecg_nn < 5), 'ecg_nn_amendsumeffort_agreement_relaxed'].mean().round(3)}")
print(f"% Sleep Only: agreement all data icu     : {data.loc[(data.population == 'icu') & (data.stage_pred_ecg_nn < 5), 'ecg_nn_amendsumeffort_agreement_relaxed'].mean().round(3)}")


for population_sel in ['sleeplab', 'icu']:
    print(f'\n{population_sel}')
    for stage_pred_version in ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']:
        print(f'\nRelaxed agreement of all epochs classified as a certain sleepstage for model {stage_pred_version}')
        for stage_no in range(1,6):
            print(f"{stages_to_names(pd.Series([stage_no]))[0]:>2}: {data.loc[(data.population == population_sel) & (data[stage_pred_version] == stage_no), 'ecg_nn_amendsumeffort_agreement_relaxed'].mean().round(3)}")


Exact stage agreement:
% agreement all data sleeplab: 0.424
% agreement all data icu     : 0.306
% Sleep Only: agreement all data sleeplab: 0.44
% Sleep Only: agreement all data icu     : 0.21

Agreement of all epochs classified as a certain sleepstage for model stage_pred_ecg_nn
N3: 0.137
N2: 0.625
N1: 0.462
 R: 0.548
 W: 0.392

Agreement of all epochs classified as a certain sleepstage for model stage_pred_amendsumeffort
N3: 0.647
N2: 0.354
N1: 0.262
 R: 0.513
 W: 0.682

Agreement of all epochs classified as a certain sleepstage for model stage_pred_ecg_nn
N3: 0.094
N2: 0.33
N1: 0.264
 R: 0.314
 W: 0.357

Agreement of all epochs classified as a certain sleepstage for model stage_pred_amendsumeffort
N3: 0.222
N2: 0.045
N1: 0.134
 R: 0.121
 W: 0.691

Relaxed stage agreement:
% agreement all data sleeplab: 0.755
% agreement all data icu     : 0.501
% Sleep Only: agreement all data sleeplab: 0.793
% Sleep Only: agreement all data icu     : 0.467

sleeplab

Relaxed agreement of all epochs

In [137]:
from sklearn.metrics import confusion_matrix
import itertools

In [9]:
def plot_confusionmatrix(cm_categories, ax):

    ax.imshow(cm_categories, interpolation='nearest', cmap=cmaps[i_axis], aspect="equal") # auto
    ax.set_yticks(range(len(categorie_names_y)));
    ax.set_xticks(range(len(categorie_names_x)));
    ax.set_xticklabels(categorie_names_x)

    ax.set_yticklabels(categorie_names_y,rotation=90, va='center', ha="right")
    ax.set_ylabel('Breathing Model Prediction')
    if iAxis>0:    
        ax.set_ylabel('')
#         ax.set_yticklabels([])

    ax.set_xlabel('HRV Model Prediction')
#     ax.set_title(setting[iN])

    thresh = cm_categories.max() / 1.3
    for i, j in itertools.product(range(cm_categories.shape[0]), range(cm_categories.shape[1])):
        plt.text(j, i, "{:0.1f}".format(cm_categories[i, j]),
                     horizontalalignment="center", verticalalignment="center",
                     color="white" if cm_categories[i, j] > thresh else "black", transform=ax.transData)
        
cmaps = ['Oranges', 'Blues']
cmaps = ['Blues', 'Blues']

In [184]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3))

i_axis = 0
data_sel = data[data.population == 'sleeplab']
cm = confusion_matrix(data_sel.stage_pred_ecg_nn, data_sel.stage_pred_amendsumeffort, normalize='true')*100
categorie_names_x = stages_names
categorie_names_y = stages_names
plot_confusionmatrix(cm, ax[i_axis])
ax[i_axis].set_title('Sleeplab')

i_axis = 1
data_sel = data[data.population == 'icu']
cm = confusion_matrix(data_sel.stage_pred_ecg_nn, data_sel.stage_pred_amendsumeffort, normalize='true')*100
categorie_names_x = stages_names
categorie_names_y = stages_names
plot_confusionmatrix(cm, ax[i_axis])
ax[i].set_title('ICU')
# plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 1.0, 'ICU')

In [196]:
alpha = 0.01
s = 6
population_plot = 'all'

fig, ax = plt.subplots(5, 2, figsize=(7,10))
ax = ax.flatten()

i = 0
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N3_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N3')

i = 1
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N3_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N3')
ax[i].legend([])

i = 2
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N2_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N2')
ax[i].legend([])

i = 3
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N2_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N2')
ax[i].legend([])

i = 4
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N1_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N1')
ax[i].legend([])

i = 5
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N1_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N1')
ax[i].legend([])

i = 6
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_R_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-R')
ax[i].legend([])

i = 7
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_R_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-R')
ax[i].legend([])

i = 8
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_W_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-Wake')
ax[i].legend([])

i = 9
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_W_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-Wake')
ax[i].legend([])

plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [224]:
alpha = 0.01
s = 6
population_plot = 'sleeplab'

fig, ax = plt.subplots(5, 2, figsize=(7,10))
ax = ax.flatten()

i = 0
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N3_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N3')

i = 1
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N3_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N3')
ax[i].legend([])

i = 2
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N2_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N2')
ax[i].legend([])

i = 3
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N2_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N2')
ax[i].legend([])

i = 4
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N1_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N1')
ax[i].legend([])

i = 5
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N1_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N1')
ax[i].legend([])

i = 6
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_R_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-R')
ax[i].legend([])

i = 7
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_R_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-R')
ax[i].legend([])

i = 8
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_W_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-Wake')
ax[i].legend([])

i = 9
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_W_agreementrelaxed_sleeplab'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-Wake')
ax[i].legend([])

plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [223]:
alpha = 0.01
s = 6
population_plot = 'icu'

fig, ax = plt.subplots(5, 2, figsize=(7,10))
ax = ax.flatten()

i = 0
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N3_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N3')

i = 1
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N3_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N3')
ax[i].legend([])

i = 2
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N2_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N2')
ax[i].legend([])

i = 3
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N2_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N2')
ax[i].legend([])

i = 4
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_N1_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-N1')
ax[i].legend([])

i = 5
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_N1_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-N1')
ax[i].legend([])

i = 6
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_R_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-R')
ax[i].legend([])

i = 7
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_R_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-R')
ax[i].legend([])

i = 8
variant = 'ecg_nn_lstm'
hue_version = 'ecg_nn_W_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title('HRV-Wake')
ax[i].legend([])

i = 9
variant = 'amendsumeffort_lstm'
hue_version = 'amendsumeffort_W_agreementrelaxed_population'
data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
ax[i] = u_map_plot(ax[i], data_plot, hue, variant, hue_order, alpha=alpha, s=s)
ax[i].set_title(population_plot)
ax[i].set_title('Breathing-Wake')
ax[i].legend([])

plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [53]:
def subplot_grid_selected_population():
    
    i = 0
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N3_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j] = u_map_plot(ax[i, j], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j].text(0.5, 0.95, 'HRV-N3', verticalalignment='center', horizontalalignment='center', transform=ax[i, j].transAxes, color='black')

    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N3_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j+1] = u_map_plot(ax[i, j+1], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j+1].text(0.5, 0.95, 'Breathing-N3', verticalalignment='center', horizontalalignment='center', transform=ax[i, j+1].transAxes, color='black')

    i = 1
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N2_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j] = u_map_plot(ax[i, j], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j].text(0.5, 0.95, 'HRV-N2', verticalalignment='center', horizontalalignment='center', transform=ax[i, j].transAxes, color='black')

    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N2_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j+1] = u_map_plot(ax[i, j+1], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j+1].text(0.5, 0.95, 'Breathing-N2', verticalalignment='center', horizontalalignment='center', transform=ax[i, j+1].transAxes, color='black')

    i = 2
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_N1_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j] = u_map_plot(ax[i, j], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j].text(0.5, 0.95, 'HRV-N1', verticalalignment='center', horizontalalignment='center', transform=ax[i, j].transAxes, color='black')

    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_N1_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j+1] = u_map_plot(ax[i, j+1], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j+1].text(0.5, 0.95, 'Breathing-N1', verticalalignment='center', horizontalalignment='center', transform=ax[i, j+1].transAxes, color='black')

    i = 3
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_R_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j] = u_map_plot(ax[i, j], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j].text(0.5, 0.95, 'HRV-R', verticalalignment='center', horizontalalignment='center', transform=ax[i, j].transAxes, color='black')

    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_R_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j+1] = u_map_plot(ax[i, j+1], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j+1].set_title(population_plot)
    ax[i, j+1].text(0.5, 0.95, 'Breathing-R', verticalalignment='center', horizontalalignment='center', transform=ax[i, j+1].transAxes, color='black')

    i = 4
    variant = 'ecg_nn_lstm'
    hue_version = 'ecg_nn_W_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j] = u_map_plot(ax[i, j], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j].text(0.5, 0.95, 'HRV-Wake', verticalalignment='center', horizontalalignment='center', transform=ax[i, j].transAxes, color='black')

    variant = 'amendsumeffort_lstm'
    hue_version = 'amendsumeffort_W_agreementrelaxed_sleeplab'
    data_plot, hue, hue_order = select_data_plot(data, population=population_plot, hue=hue_version)
    ax[i, j+1] = u_map_plot(ax[i, j+1], data_plot, hue, variant, hue_order, alpha=alpha, s=s, plot_type = plot_type, legend=False)
    ax[i, j+1].text(0.5, 0.95, 'Breathing-Wake', verticalalignment='center', horizontalalignment='center', transform=ax[i, j+1].transAxes, color='black')

    return ax


def axis_formatting_5x4_grid(ax):

    j = 0
    for i in range(0,5):
        ax[i, j+1].set_xlabel('')
        ax[i, j+1].set_ylabel('')
        ax[i, j].set_xlabel('')
        ax[i, j].set_ylabel('UMAP C2')

        ax[i, j].set_xticks(hrv_xticks)
        ax[i, j].set_yticks(hrv_yticks)
        ax[i, j+1].set_xticks(breathing_xticks)
        ax[i, j+1].set_yticks(breathing_yticks)

        ax[i, j].tick_params(length=2, pad=-11)
        ax[i, j+1].tick_params(length=2, pad=-11)

    for j in range(4):
        ax[-1, j].set_xlabel('UMAP C1')

    j = 2

    for i in range(0,5):
        ax[i, j+1].set_xlabel('')
        ax[i, j+1].set_ylabel('')
        ax[i, j].set_xlabel('')
        ax[i, j].set_ylabel('')

        ax[i, j].set_xticks(hrv_xticks)
        ax[i, j].set_yticks(hrv_yticks)
        ax[i, j+1].set_xticks(breathing_xticks)
        ax[i, j+1].set_yticks(breathing_yticks)

        ax[i, j].tick_params(length=2, pad=-11)
        ax[i, j+1].tick_params(length=2, pad=-11)

    for j in range(4):
        ax[-1, j].set_xlabel('UMAP C1')
        
    return ax


hrv_xticks = [0, 7, 14]
hrv_yticks = [0, 7, 14]
breathing_xticks = [0, 7, 14]
breathing_yticks = [-14, -5, 4]

alpha = 0.01
s = 6
plot_type = 'scatter'

fig, ax = plt.subplots(5, 4, figsize=(10,10))

j = 0
population_plot = 'sleeplab'
ax = subplot_grid_selected_population()

j = 2
population_plot = 'icu'
ax = subplot_grid_selected_population()
    
colors = ['red', 'green']
lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
labels = ['Disagreement', 'Agreement']
# plt.legend(lines, labels)

ax[0, 0].legend(lines, labels, 
             frameon=False, markerscale=1, ncol=1, bbox_to_anchor=(0.25, 0.96, 0.5, 0.5), loc='center')

margin = 0.05
plt.subplots_adjust(left=margin, bottom=margin, right=1-margin, top=1-margin, wspace=0, hspace=0)

ax[0,0].text(0.27, 0.97, 'Sleeplab',
        verticalalignment='center', horizontalalignment='center',
        transform=fig.transFigure,
        color='black')
ax[2,0].text(0.73, 0.97, 'ICU',
        verticalalignment='center', horizontalalignment='center',
        transform=fig.transFigure,
        color='black')

ax = axis_formatting_5x4_grid(ax)

plt.tight_layout()

plt.savefig('population_modality_stage_agreement_scatter.svg')
plt.savefig('population_modality_stage_agreement_scatter.jpg', dpi=500)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:123: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:155: UserWarning: Tight layout not applied. tight_layout cannot make axes width small enough to accommodate all axes decorations


In [54]:
plot_type = 'kde'
alpha = 0.5

fig, ax = plt.subplots(5, 4, figsize=(10,10))

j = 0
population_plot = 'sleeplab'
ax = subplot_grid_selected_population()

j = 2
population_plot = 'icu'
ax = subplot_grid_selected_population()
    
colors = ['red', 'green']
lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
labels = ['Disagreement', 'Agreement']
# plt.legend(lines, labels)

ax[0, 0].legend(lines, labels, 
             frameon=False, markerscale=1, ncol=1, bbox_to_anchor=(0.25, 0.96, 0.5, 0.5), loc='center')

margin = 0.05
plt.subplots_adjust(left=margin, bottom=margin, right=1-margin, top=1-margin, wspace=0, hspace=0)

ax[0,0].text(0.27, 0.97, 'Sleeplab',
        verticalalignment='center', horizontalalignment='center',
        transform=fig.transFigure,
        color='black')
ax[2,0].text(0.73, 0.97, 'ICU',
        verticalalignment='center', horizontalalignment='center',
        transform=fig.transFigure,
        color='black')

ax = axis_formatting_5x4_grid(ax)

plt.tight_layout()

plt.savefig('population_modality_stage_agreement_kde.svg')
plt.savefig('population_modality_stage_agreement_kde.jpg', dpi=500)


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  after removing the cwd from sys.path.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: UserWarning: Tight layout not applied. tight_layout cannot make axes width small enough to accommodate all axes decorations


In [15]:
[x for x in data.columns if 'umap' in x]

['umap_ecg_nn_lstm_2d_0',
 'umap_ecg_nn_lstm_2d_1',
 'umap_amendsumeffort_lstm_2d_0',
 'umap_amendsumeffort_lstm_2d_1',
 'umap_ecg_and_breathing_lstm_2d_0',
 'umap_ecg_and_breathing_lstm_2d_1']

In [214]:
hue_order

['Agreement', 'Disagreement']

In [215]:
hue.unique()

array(['Agreement', 'Disagreement'], dtype=object)

In [217]:
hue.unique()

array(['Agreement', 'Disagreement'], dtype=object)

In [218]:
variant

'ecg_nn_lstm'

### add the agreement info to the h5 files

In [16]:
data[['ecg_nn_amendsumeffort_agreement', 'ecg_nn_amendsumeffort_agreement_relaxed']] = data[['ecg_nn_amendsumeffort_agreement', 'ecg_nn_amendsumeffort_agreement_relaxed']].astype(int)

In [45]:
import re

# dir_sleeplab = f'/scr/wolfgang/Sleep_And_Breathing/sleeplab_files_step5'
# dir_icu = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step5_3'
dir_icu = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step7'

In [46]:
data.loc[data.population == 'sleeplab', 'study_id'];

In [47]:
for population in ['icu']: # ['sleeplab', 'icu']

    if population == 'sleeplab':
        input_dir = dir_sleeplab
        savedir = f'/scr/wolfgang/Sleep_And_Breathing/sleeplab_files_step6'

    elif population == 'icu':
        input_dir = dir_icu
#         savedir = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step6'
        savedir = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step8'

    if not os.path.exists(savedir):
        os.makedirs(savedir)
        
    files = os.listdir(input_dir)
    files.sort()
    
    for file in files:
        study_id = int(re.search('\d\d\d', file)[0])
        data_10hz, hdr, fs = read_in_routine(os.path.join(input_dir, file))
        data_agreement = data.loc[(data.population==population) & (data.study_id==study_id)].copy()
        data_agreement.set_index('datetime', inplace=True)
        assert np.all(np.isin(data_agreement.index, data_10hz.index)), 'More data in hlh data?'
        agreement_cols = ['ecg_nn_amendsumeffort_agreement', 'ecg_nn_amendsumeffort_agreement_relaxed']
        for col_tmp in agreement_cols:
            if col_tmp in data_10hz.columns:
                data_10hz.drop([col_tmp], axis=1, inplace=True)
        
        data_10hz = data_10hz.join(data_agreement[agreement_cols])
        data_10hz[agreement_cols] = data_10hz[agreement_cols].fillna(method='ffill', axis=0, limit=30*fs)
        
        write_to_hdf5_file(data_10hz, os.path.join(savedir, file), hdr)

ValueError: Data already exists. Set Overwrite==True to remove old data.

In [41]:
data_10hz, hdr, fs = read_in_routine(os.path.join(input_dir, 'icusleep_122.h5'))


In [42]:
file

'icusleep_122.h5'

In [43]:
input_dir

'/scr/wolfgang/Sleep_And_Breathing/icu_files_step7'

In [44]:
data_10hz['ecg_nn_amendsumeffort_agreement'].dropna()

Series([], Freq: 100L, Name: ecg_nn_amendsumeffort_agreement, dtype: float32)

In [33]:
data_agreement['ecg_nn_amendsumeffort_agreement'].dropna()

datetime
2018-06-06 15:29:30    1
2018-06-06 15:30:00    1
2018-06-06 15:30:30    1
2018-06-06 15:31:00    1
2018-06-06 15:31:30    1
                      ..
2018-06-07 04:21:30    0
2018-06-07 04:22:00    0
2018-06-07 04:22:30    0
2018-06-07 04:23:00    0
2018-06-07 04:23:30    0
Name: ecg_nn_amendsumeffort_agreement, Length: 1195, dtype: int64

In [34]:
data_agreement['ecg_nn_amendsumeffort_agreement_relaxed'].dropna()

datetime
2018-06-06 15:29:30    1
2018-06-06 15:30:00    1
2018-06-06 15:30:30    1
2018-06-06 15:31:00    1
2018-06-06 15:31:30    1
                      ..
2018-06-07 04:21:30    0
2018-06-07 04:22:00    0
2018-06-07 04:22:30    0
2018-06-07 04:23:00    0
2018-06-07 04:23:30    0
Name: ecg_nn_amendsumeffort_agreement_relaxed, Length: 1195, dtype: int64

In [27]:
[x for x in data_agreement.columns if 'dat' in x.lower()]

[]

In [23]:
data_agreement.columns

Index(['h_lstm_activity10sec_0', 'h_lstm_activity10sec_1',
       'h_lstm_activity10sec_2', 'h_lstm_activity10sec_3',
       'h_lstm_activity10sec_4', 'h_lstm_activity10sec_5',
       'h_lstm_activity10sec_6', 'h_lstm_activity10sec_7',
       'h_lstm_activity10sec_8', 'h_lstm_activity10sec_9',
       ...
       'yp_lstm_ecg_nn_0', 'yp_lstm_ecg_nn_1', 'yp_lstm_ecg_nn_2',
       'yp_lstm_ecg_nn_3', 'yp_lstm_ecg_nn_4', 'stage_pred_ecg_nn', 'study_id',
       'population', 'ecg_nn_amendsumeffort_agreement',
       'ecg_nn_amendsumeffort_agreement_relaxed'],
      dtype='object', length=325)